In [1]:
import json
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

train = pd.read_csv("../data/processed/train_FD001_processed.csv")
val = pd.read_csv("../data/processed/val_FD001_processed.csv")

with open("../data/processed/feature_cols.json") as f:
    feature_cols = json.load(f)

X_train, y_train = train[feature_cols], train["RUL_clipped"]
X_val, y_val = val[feature_cols], val["RUL_clipped"]

X_test = pd.read_csv("../data/processed/X_test_FD001.csv")
y_test = pd.read_csv("../data/processed/y_test_FD001.csv")

print(X_train.shape, X_val.shape)
print(X_test.shape, y_test.shape)
print(X_test.iloc[:2, :3])
print(y_test.head(3))

(16561, 45) (4070, 45)
(100, 45) (100, 1)
   sensor_2  sensor_3  sensor_4
0 -0.195953 -1.523214 -1.111959
1 -0.256180 -0.640748  0.217605
     0
0  112
1   98
2   69


In [2]:
# To Create the Baseline model 
y_test = y_test.iloc[:, 0]

def rmse(y_true, y_pred):
    return np.sqrt(np.mean((y_pred - y_true) ** 2))

def mae(y_true, y_pred):
    return np.mean(np.abs(y_pred - y_true))

def nasa_score(y_true, y_pred):
    d = np.asarray(y_pred, dtype=float) - np.asarray(y_true, dtype=float)
    return np.sum(np.where(d < 0, np.exp(-d / 13) - 1, np.exp(d / 10) - 1))

def evaluate(y_true, y_pred):
    return {"RMSE": rmse(y_true, y_pred), "MAE": mae(y_true, y_pred),
            "NASA": nasa_score(y_true, y_pred)}

baseline_pred = np.full(len(y_val), y_train.mean())
print("y_test shape:", y_test.shape)
print("train mean RUL:", round(y_train.mean(), 1))
print(evaluate(y_val, baseline_pred))

y_test shape: (100,)
train mean RUL: 87.0
{'RMSE': np.float64(41.72082538935907), 'MAE': np.float64(36.929802663876686), 'NASA': np.float64(1285855.8101210925)}


In [3]:
# Linear Regression
from sklearn.linear_model import LinearRegression

# Create the model
lr_model = LinearRegression()

# Train the model
lr_model.fit(X_train, y_train)

# Predict RUL for validation data
lr_val_pred = lr_model.predict(X_val)

# Evaluate the model
print("Linear Regression Results:")
print(evaluate(y_val, lr_val_pred))

Linear Regression Results:
{'RMSE': np.float64(18.714693616699716), 'MAE': np.float64(15.220399740249164), 'NASA': np.float64(24401.959799286782)}


In [4]:
# Rigge
from sklearn.linear_model import Ridge

# Create the Ridge Regression model
ridge_model = Ridge(alpha=1.0)

# Train the model
ridge_model.fit(X_train, y_train)

# Predict RUL for validation data
ridge_val_pred = ridge_model.predict(X_val)

# Evaluate the model
print("Ridge Regression Results:")
print(evaluate(y_val, ridge_val_pred))

Ridge Regression Results:
{'RMSE': np.float64(18.711399291401218), 'MAE': np.float64(15.2170759553133), 'NASA': np.float64(24373.73673866136)}


In [5]:
# Random Forest 
from sklearn.ensemble import RandomForestRegressor

# Create the model
rf_model = RandomForestRegressor(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)

# Train the model
rf_model.fit(X_train, y_train)

# Predict RUL for validation data
rf_val_pred = rf_model.predict(X_val)

# Evaluate the model
print("Random Forest Results:")
print(evaluate(y_val, rf_val_pred))

Random Forest Results:
{'RMSE': np.float64(17.001331786249644), 'MAE': np.float64(11.938432432432432), 'NASA': np.float64(31375.160207824993)}


In [7]:
# Extra resgressor
from sklearn.ensemble import ExtraTreesRegressor

# Create the model
et_model = ExtraTreesRegressor(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)

# Train the model
et_model.fit(X_train, y_train)

# Predict RUL for validation data
et_val_pred = et_model.predict(X_val)

# Evaluate the model
print("Extra Trees Results:")
print(evaluate(y_val, et_val_pred))


Extra Trees Results:
{'RMSE': np.float64(15.743005890608835), 'MAE': np.float64(11.33060687960688), 'NASA': np.float64(22045.22134915092)}


In [8]:
# Hisgradient 
from sklearn.ensemble import HistGradientBoostingRegressor

# Create the model
hgb_model = HistGradientBoostingRegressor(
    max_iter=100,
    learning_rate=0.1,
    random_state=42
)

# Train the model
hgb_model.fit(X_train, y_train)

# Predict RUL for validation data
hgb_val_pred = hgb_model.predict(X_val)

# Evaluate the model
print("HistGradientBoosting Results:")
print(evaluate(y_val, hgb_val_pred))



c:\Users\lenovo\anaconda3\Lib\site-packages\joblib\externals\loky\backend\context.py:131: UserWarning: Could not find the number of physical cores for the following reason:
[WinError 2] The system cannot find the file specified
Returning the number of logical cores instead. You can silence this warning by setting LOKY_MAX_CPU_COUNT to the number of cores you want to use.
  warnings.warn(
  File "c:\Users\lenovo\anaconda3\Lib\site-packages\joblib\externals\loky\backend\context.py", line 247, in _count_physical_cores
    cpu_count_physical = _count_physical_cores_win32()
  File "c:\Users\lenovo\anaconda3\Lib\site-packages\joblib\externals\loky\backend\context.py", line 299, in _count_physical_cores_win32
    cpu_info = subprocess.run(
        "wmic CPU Get NumberOfCores /Format:csv".split(),
        capture_output=True,
        text=True,
    )
  File "c:\Users\lenovo\anaconda3\Lib\subprocess.py", line 554, in run
    with Popen(*popenargs, **kwargs) as process:
         ~~~~~^^^^^^^^^^^

HistGradientBoosting Results:
{'RMSE': np.float64(15.89573020555005), 'MAE': np.float64(11.242940186084526), 'NASA': np.float64(23786.50022589285)}
